In [ ]:
import pandas as pd


In [ ]:
data1 = pd.read_csv('/content/drive/MyDrive/Training_Data_Lung_Dieses/Healthcare.csv')
data2 = pd.read_csv('/content/drive/MyDrive/Training_Data_Lung_Dieses/respiratory symptoms and treatment.csv')
data3 = pd.read_csv('/content/drive/MyDrive/Training_Data_Lung_Dieses/Symptom2Disease.csv')
0
print('Healthcare.csv Head:')
print(data1.shape)
print(data1.head())
print('\nrespiratory symptoms and treatment.csv Head:')
print(data2.shape)
print(data2.head())
print('\nSymptom2Disease.csv Head:')
print(data3.shape)
print(data3.head())

Healthcare.csv Head:
(25000, 6)
   Patient_ID  Age  Gender                                           Symptoms  \
0           1   29    Male              fever, back pain, shortness of breath   
1           2   76  Female                   insomnia, back pain, weight loss   
2           3   78    Male                    sore throat, vomiting, diarrhea   
3           4   58   Other  blurred vision, depression, weight loss, muscl...   
4           5   55  Female                    swelling, appetite loss, nausea   

   Symptom_Count           Disease  
0              3           Allergy  
1              3  Thyroid Disorder  
2              3         Influenza  
3              4            Stroke  
4              3     Heart Disease  

respiratory symptoms and treatment.csv Head:
(38537, 6)
                     Symptoms  Age     Sex Disease    Treatment Nature
0                   coughing   5.0  female  Asthma   Omalizumab   high
1  tight feeling in the chest  4.0  female  Asthma  Mepolizu

In [ ]:
print('data1 shape final: ',data1.loc[data1['Disease'].isin(data2['Disease'].unique())].shape)
print('data3 shape final: ',data3.loc[data3['label'].isin(data2['Disease'].unique())].shape)


data1 shape final:  (3250, 6)
data3 shape final:  (50, 3)


In [ ]:
df_symptoms_data1 = data1[['Symptoms', 'Disease']]
df_symptoms_data2 = data2[['Symptoms', 'Disease']]
df_symptoms_data3 = data3.rename(columns={'text': 'Symptoms', 'label': 'Disease'})[['Symptoms', 'Disease']]

# Concatenate all three dataframes
symptoms_df = pd.concat([df_symptoms_data1, df_symptoms_data2, df_symptoms_data3], ignore_index=True)

# Filter symptoms_df to include only diseases present in data2
symptoms_df = symptoms_df[symptoms_df['Disease'].isin(data2['Disease'].unique())]

print("Combined Symptoms and Disease DataFrame shape (filtered by data2 diseases):", symptoms_df.shape)
display(symptoms_df.head())

Combined Symptoms and Disease DataFrame shape (filtered by data2 diseases): (41837, 2)


,Symptoms,Disease
2,"sore throat, vomiting, diarrhea",Influenza
23,"weight gain, sore throat, back pain, sneezing",Influenza
28,"cough, tremors, dizziness",Asthma
34,"sweating, fever, sore throat, swelling, diarrh...",Influenza
41,"runny nose, dizziness, swelling, sore throat, ...",Tuberculosis


In [ ]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

symptoms_df["Label"] = encoder.fit_transform(symptoms_df["Disease"])

print(symptoms_df.head())

                                             Symptoms       Disease  Label
2                     sore throat, vomiting, diarrhea     Influenza      7
23      weight gain, sore throat, back pain, sneezing     Influenza      7
28                          cough, tremors, dizziness        Asthma      3
34  sweating, fever, sore throat, swelling, diarrh...     Influenza      7
41  runny nose, dizziness, swelling, sore throat, ...  Tuberculosis     13


In [ ]:
from sklearn.model_selection import train_test_split

# Drop rows with any NaN values from symptoms_df before splitting
symptoms_df_cleaned = symptoms_df.dropna(subset=['Symptoms', 'Disease'])

# Assuming symptoms_df_cleaned is your DataFrame and 'Disease' is your target variable
X = symptoms_df_cleaned['Symptoms']
y = symptoms_df_cleaned['Label']

# Split into 70% training and 30% temporary (for validation and test)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)

# Split temporary into 15% validation and 15% test (50% of temporary for each)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print(f"Training set size: {len(X_train)} ({len(X_train)/len(symptoms_df_cleaned):.2%})")
print(f"Validation set size: {len(X_val)} ({len(X_val)/len(symptoms_df_cleaned):.2%})")
print(f"Testing set size: {len(X_test)} ({len(X_test)/len(symptoms_df_cleaned):.2%})")

Training set size: 28695 (70.00%)
Validation set size: 6149 (15.00%)
Testing set size: 6149 (15.00%)


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-uncased"
)

train_encodings = tokenizer(
    X_train.tolist(),
    truncation=True,
    padding=True,
    max_length=64
)

test_encodings = tokenizer(
    X_val.tolist(),
    truncation=True,
    padding=True,
    max_length=64
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
import torch

class DiseaseDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):

        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):

        item = {}

        for key, value in self.encodings.items():

            item[key] = torch.tensor(value[idx])

        item["labels"] = torch.tensor(
            self.labels.iloc[idx]
        )

        return item

    def __len__(self):

        return len(self.labels)

In [ ]:
train_dataset = DiseaseDataset(
    train_encodings,
    y_train
)

test_dataset = DiseaseDataset(
    test_encodings,
    y_val
)

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(encoder.classes_)
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(

    output_dir="./results",

    eval_strategy="epoch",

    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=16,

    per_device_eval_batch_size=16,

    num_train_epochs=5,

    weight_decay=0.01,

    logging_steps=10,

    load_best_model_at_end=True
)

In [ ]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.6 MB/s eta 0:00:00


In [ ]:
import evaluate
import numpy as np
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report

accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)

    # Calculate accuracy
    accuracy = accuracy_metric.compute(predictions=predictions, references=labels)

    # Calculate F1, Precision, Recall (macro average for multi-class)
    f1 = f1_score(labels, predictions, average='macro')
    precision = precision_score(labels, predictions, average='macro')
    recall = recall_score(labels, predictions, average='macro')

    # Print full classification report (optional, can be removed if too verbose during training)
    print("\nClassification Report:\n")
    print(classification_report(labels, predictions, target_names=encoder.classes_))

    return {
        "accuracy": accuracy["accuracy"],
        "f1_score_macro": f1,
        "precision_macro": precision,
        "recall_macro": recall
    }

In [ ]:
from transformers import Trainer

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=test_dataset,

    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


In [ ]:
trainer.save_model("lung_model")

tokenizer.save_pretrained("lung_model")

In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import accuracy_score
import torch.nn.functional as F
import numpy as np

# Define a simple Dataset for inference
class InferenceDataset(Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels.iloc[idx])
        return item

    def __len__(self):
        return len(self.encodings['input_ids'])

# Tokenize X_test
inputs_test = tokenizer(
    X_test.tolist(),
    truncation=True,
    padding=True,
    max_length=64,
    return_tensors="pt"
)

# Create a dataset and DataLoader for X_test, including y_test labels
inference_dataset = InferenceDataset(inputs_test, y_test)

# Adjust batch size based on GPU memory. Start with a conservative value.
batch_size = 32 # You might need to adjust this value
inference_loader = DataLoader(inference_dataset, batch_size=batch_size, shuffle=False)

# Move model to appropriate device (CPU or GPU)
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)

model.eval()

all_predictions = []
all_true_labels = []
all_logits = []

with torch.no_grad():
    for batch in inference_loader:
        # Move input tensors and labels for the current batch to the same device as the model
        batch_inputs = {key: value.to(device) for key, value in batch.items() if key != 'labels'}
        batch_labels = batch['labels'].to(device) # Keep labels on the device for loss calculation

        outputs = model(**batch_inputs)
        logits = outputs.logits
        prediction = logits.argmax(-1)

        all_predictions.extend(prediction.cpu().numpy())
        all_true_labels.extend(batch_labels.cpu().numpy())
        all_logits.append(logits.cpu()) # Collect logits from CPU to avoid OOM if all_logits become too large

# Convert collected lists to numpy arrays
final_predictions = np.array(all_predictions)
final_true_labels = np.array(all_true_labels)
final_logits = torch.cat(all_logits, dim=0)

# Calculate accuracy
accuracy = accuracy_score(final_true_labels, final_predictions)

# Calculate Cross-Entropy Loss
# F.cross_entropy expects logits and true labels
# Make sure final_true_labels is a long tensor for cross_entropy
loss = F.cross_entropy(final_logits, torch.tensor(final_true_labels, dtype=torch.long))

print(f"Accuracy on X_test: {accuracy:.4f}")
print(f"Cross-Entropy Loss on X_test: {loss.item():.4f}")

print("\nExample predictions (first 10) and true labels:")
for i in range(10):
    predicted_disease = encoder.inverse_transform([final_predictions[i]])[0]
    true_disease = encoder.inverse_transform([final_true_labels[i]])[0]
    print(f"Sample {i+1}: Predicted='{predicted_disease}', True='{true_disease}'")


In [ ]:
probabilities = torch.softmax(
    outputs.logits,
    dim=1
)

print(probabilities)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("/content/drive/MyDrive/results/checkpoint-8970")

# Load model
model = AutoModelForSequenceClassification.from_pretrained("/content/drive/MyDrive/results/checkpoint-8970")

# Move model to GPU (if available)
model.to(device)

model.eval()

In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import accuracy_score
import torch.nn.functional as F
import numpy as np

# Define a simple Dataset for inference
class InferenceDataset(Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels.iloc[idx])
        return item

    def __len__(self):
        return len(self.encodings['input_ids'])

# Tokenize X_test
inputs_test = tokenizer(
    X_test.tolist(),
    truncation=True,
    padding=True,
    max_length=64,
    return_tensors="pt"
)

# Create a dataset and DataLoader for X_test, including y_test labels
inference_dataset = InferenceDataset(inputs_test, y_test)

# Adjust batch size based on GPU memory. Start with a conservative value.
batch_size = 32 # You might need to adjust this value
inference_loader = DataLoader(inference_dataset, batch_size=batch_size, shuffle=False)

# Move model to appropriate device (CPU or GPU)
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)

model.eval()

all_predictions = []
all_true_labels = []
all_logits = []

with torch.no_grad():
    for batch in inference_loader:
        # Move input tensors and labels for the current batch to the same device as the model
        batch_inputs = {key: value.to(device) for key, value in batch.items() if key != 'labels'}
        batch_labels = batch['labels'].to(device) # Keep labels on the device for loss calculation

        outputs = model(**batch_inputs)
        logits = outputs.logits
        prediction = logits.argmax(-1)

        all_predictions.extend(prediction.cpu().numpy())
        all_true_labels.extend(batch_labels.cpu().numpy())
        all_logits.append(logits.cpu()) # Collect logits from CPU to avoid OOM if all_logits become too large

# Convert collected lists to numpy arrays
final_predictions = np.array(all_predictions)
final_true_labels = np.array(all_true_labels)
final_logits = torch.cat(all_logits, dim=0)

# Calculate accuracy
accuracy = accuracy_score(final_true_labels, final_predictions)

# Calculate Cross-Entropy Loss
# F.cross_entropy expects logits and true labels
# Make sure final_true_labels is a long tensor for cross_entropy
loss = F.cross_entropy(final_logits, torch.tensor(final_true_labels, dtype=torch.long))

print(f"Accuracy on X_test: {accuracy:.4f}")
print(f"Cross-Entropy Loss on X_test: {loss.item():.4f}")

print("\nExample predictions (first 10) and true labels:")
for i in range(10):
    predicted_disease = encoder.inverse_transform([final_predictions[i]])[0]
    true_disease = encoder.inverse_transform([final_true_labels[i]])[0]
    print(f"Sample {i+1}: Predicted='{predicted_disease}', True='{true_disease}'")
